# Day 6: Streamlit UI with Groq API integration

Welcome to Day 6 of the AI Foundations & LLM Fundamentals phase! Today, we transition from backend logic and terminal outputs to building interactive frontends. We will construct a real-time web interface using **Streamlit** and connect it to the **Groq API** to achieve sub-second inference speeds.

## Core Theory (Just-in-Time)

### Why Streamlit & Groq?
- **Rapid Prototyping (Streamlit):** Traditional full-stack development introduces significant overhead. Streamlit allows you to build data-driven web applications entirely in Python, seamlessly handling frontend state and reactive UI updates.
- **Sub-Second Latency (Groq):** Groq's Language Processing Units (LPUs) execute open-source LLMs (like LLaMA 3 and Mixtral) at blazing-fast speeds. Low latency is critical for conversational UIs to prevent user experience degradation.

### AI Security Implications in Frontend Design
When building a user-facing AI application, security must be considered at the frontend level before the request even reaches the LLM:
1. **PII Protection:** Users often inadvertently paste sensitive data (emails, API keys, SSNs). A production UI should implement basic pre-processing (regex or lightweight NLP) to redact Personally Identifiable Information (PII) before sending it to the API.
2. **Prompt Injection Mitigation:** While robust mitigation often happens server-side, basic sanitization (stripping unusual characters, limiting input length) at the UI level reduces surface area for attacks.
3. **Graceful Fallbacks:** External APIs fail (rate limits, timeouts). Your UI must gracefully handle these exceptions and display user-friendly fallback messages instead of exposing stack traces or leaving the user hanging.


In [1]:
# BASIC TIER: Isolate the core concept (Calling Groq API)
import os
from groq import Groq

# Using a try/except block to bypass authentication issues during local notebook validation
try:
    client = Groq(api_key=os.environ.get("GROQ_API_KEY", "dummy"))
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": "What is 2+2?"}]
    )
    print("Basic Response:", response.choices[0].message.content)
except Exception as e:
    print(f"Basic API Call Failed (expected during test): {e}")


Basic API Call Failed (expected during test): Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


In [2]:
# MEDIUM TIER: Emphasize clean OOP and state management
import os
import streamlit as st
from groq import Groq

class GroqChatManager:
    """Manages Groq API interactions and Streamlit state."""
    
    def __init__(self, model_name: str = "llama3-8b-8192"):
        self.model_name = model_name
        # Securely initialize client; fallback to dummy for local testing
        self.client = Groq(api_key=os.environ.get("GROQ_API_KEY", "dummy"))
        
    def get_response(self, user_input: str) -> str:
        """Fetch response from Groq API with basic error handling."""
        try:
            response = self.client.chat.completions.create(
                model=self.model_name,
                messages=[{"role": "user", "content": user_input}]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: API Call Failed - {e}"

# Streamlit mockup logic
st.session_state["dummy_input"] = "Tell me a joke."
manager = GroqChatManager()
response_text = manager.get_response(st.session_state["dummy_input"])
print("Medium Response:", response_text)


2026-08-20 12:49:40.886 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:40.888 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`


2026-08-20 12:49:40.889 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:40.931 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Medium Response: Error: API Call Failed - Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


In [3]:
# ADVANCED TIER: Production-Grade Streamlit Application
# Below is a complete, production-grade Streamlit application demonstrating OOP,
# AI Security (Fallbacks), and Streamlit state management.
# Note: To run this code, save it to `app.py` and run `streamlit run app.py`

import os
import re
import logging
import streamlit as st
from groq import Groq
from typing import List, Dict, Optional

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class SecurityFilter:
    """Handles basic PII redaction and prompt injection mitigation."""
    
    @staticmethod
    def redact_pii(text: str) -> str:
        """Redacts potential email addresses as a basic PII measure."""
        email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
        return re.sub(email_pattern, "[REDACTED EMAIL]", text)
    
    @staticmethod
    def sanitize_input(text: str) -> str:
        """Sanitizes input to mitigate basic prompt injection."""
        # A real implementation would use a more robust filtering model or LLM guardrail.
        return text.strip()

class ChatApplication:
    """Production-grade Chat Application using Groq."""
    
    def __init__(self) -> None:
        self.api_key: Optional[str] = os.environ.get("GROQ_API_KEY", "dummy")
        self.model_name: str = "llama3-8b-8192"
        self._initialize_state()

    def _initialize_state(self) -> None:
        """Initializes Streamlit session state."""
        if "messages" not in st.session_state:
            st.session_state.messages = [
                {"role": "system", "content": "You are a helpful and secure AI assistant."}
            ]

    def _get_client(self) -> Optional[Groq]:
        """Safely retrieves the Groq client."""
        if not self.api_key or self.api_key == "dummy":
            logger.warning("Invalid or missing GROQ_API_KEY.")
            return None
        return Groq(api_key=self.api_key)

    def display_history(self) -> None:
        """Renders the conversation history."""
        for message in st.session_state.messages:
            if message["role"] != "system":
                with st.chat_message(message["role"]):
                    st.markdown(message["content"])

    def handle_input(self) -> None:
        """Processes user input, applies security filters, and streams response."""
        prompt = st.chat_input("What is on your mind?")
        if not prompt:
            return

        # Security step: Sanitize and redact PII
        sanitized_prompt = SecurityFilter.sanitize_input(prompt)
        safe_prompt = SecurityFilter.redact_pii(sanitized_prompt)

        # Update and display user message
        st.session_state.messages.append({"role": "user", "content": safe_prompt})
        with st.chat_message("user"):
            st.markdown(safe_prompt)

        # Generate response
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            full_response = ""
            
            client = self._get_client()
            if not client:
                fallback_msg = "System Error: API Key missing or invalid. Please check configuration."
                st.error(fallback_msg)
                st.session_state.messages.append({"role": "assistant", "content": fallback_msg})
                return

            try:
                stream = client.chat.completions.create(
                    model=self.model_name,
                    messages=st.session_state.messages,
                    stream=True,
                    temperature=0.7,
                    max_tokens=1024,
                )
                
                for chunk in stream:
                    delta = chunk.choices[0].delta.content
                    if delta:
                        full_response += delta
                        message_placeholder.markdown(full_response + "▌")
                
                message_placeholder.markdown(full_response)
                st.session_state.messages.append({"role": "assistant", "content": full_response})
                
            except Exception as e:
                logger.error(f"API Error: {e}")
                fallback_msg = "I'm currently experiencing technical difficulties. Please try again later."
                st.error(fallback_msg)
                st.session_state.messages.append({"role": "assistant", "content": fallback_msg})

def main() -> None:
    st.set_page_config(page_title="Secure Groq Chat", page_icon="⚡")
    st.title("⚡ Secure Groq-Powered Chat Interface")
    
    app = ChatApplication()
    app.display_history()
    app.handle_input()

if __name__ == "__main__":
    main()


2026-08-20 12:49:41.218 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.219 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.345 
  command:

    streamlit run /app/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]


2026-08-20 12:49:41.346 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.347 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


...

2026-08-20 12:49:41.351 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.352 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.353 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.355 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-20 12:49:41.355 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


## Common Pitfalls

1. **State Loss on Re-render:** Streamlit re-runs the *entire script* from top to bottom every time an interaction occurs. If you do not store your chat history in `st.session_state`, it will be wiped out after every message.
2. **Blocking the Main Thread:** Making synchronous API calls without streaming can freeze the UI. Always use streaming for LLM text generation.
3. **Exposing API Keys:** Hardcoding API keys in your application code is a severe security risk. Always use environment variables (`os.environ`) or Streamlit's secrets management (`st.secrets`).
4. **Unconstrained Context Window:** `st.session_state.messages` grows indefinitely. In production, implement a sliding window or summarization strategy to truncate older messages to prevent exceeding the LLM's maximum context window.
5. **Ignoring Exceptions:** Failing to implement proper `try/except` blocks around external API calls can result in the app crashing and exposing sensitive internal logic via stack traces to the end user.

## Practical Lab / Homework

**Your Task for Today:**

1. Save the Advanced Tier code provided above to a file named `groq_app.py`.
2. Install the necessary libraries: `uv pip install streamlit groq`.
3. Set your Groq API key in your terminal session (e.g., `export GROQ_API_KEY='your_api_key'`).
4. Run the application locally using `.venv/bin/streamlit run groq_app.py`.
5. **Modification Challenge:**
   - Modify the UI to include a Streamlit sidebar (`st.sidebar`).
   - Add a dropdown selector (`st.selectbox`) that allows the user to switch between two different Groq models dynamically (e.g., `llama3-8b-8192` and `mixtral-8x7b-32768`).
   - Add a "Clear Chat" button in the sidebar that resets the `st.session_state.messages`.
6. **Video Walkthrough:** Record a brief 2-3 minute async video (using Loom or similar) explaining your design decisions, specifically focusing on how you managed Streamlit state and implemented the security/fallback mechanisms.

*Verification:* You will know you have succeeded when you can chat with the assistant, toggle the underlying model from the sidebar without losing history, and safely clear the chat.

## Reference Links

- [Streamlit Documentation: Session State](https://docs.streamlit.io/library/api-reference/session-state)
- [Groq API Documentation: Streaming](https://console.groq.com/docs/quickstart)
- [OWASP Top 10 for LLM Applications](https://owasp.org/www-project-top-10-for-large-language-model-applications/)
